In [1]:
import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
from utils import *
from build_trna_expression_reactions import charged_trna_metabolites, modified_trna_transcript_c

import build_mrna_expression_reactions as bm
from gene_information import gene_information

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


../scripts/build_trna_expression_reactions.py:48 UserWarning: Mature tRNA sequence not in the expected length range (76<=L<=93)


In [2]:
transport_translocation_atp_cost = 0.5 # 1 ATP/2 residues
proteolysis_translocation_atp_cost = 0.5 # 1 ATP/2 residues

translation_efs = ['HGNC:3189', 'HGNC:3214', 'HGNC:3208', 'HGNC:3300']
n_ub = 4 # see this - no. of ubiquitins to add to protein

# # DON'T DELETE------------------------------------------------
# import pandas as pd
# e3_ligase = pd.read_csv(local_data_path + 'raw/E3_HPA.tsv', sep = '\t')
# e3_ligase = e3_ligase[e3_ligase['RNA cell line specificity'] == 'Low cell line specificity']
# e3_ligase = e3_ligase.loc[e3_ligase[e3_ligase['Subcellular main location'].notna()].index,:]
# e3_ligase = e3_ligase.loc[[i for i in e3_ligase.index if ('Cytosol' in e3_ligase.loc[i, 'Subcellular main location'])], :]
# # e3_ligase = e3_ligase[e3_ligase['Subcellular main location'] == 'Cytosol']
# top_gene_idx = e3_ligase.iloc[:, e3_ligase.columns.tolist().index('Tissue RNA - adipose tissue [NX]'):].mean(axis = 1).sort_values(ascending = False).index.tolist()[3]
# e3_uniprot_id = e3_ligase.loc[top_gene_idx, 'Uniprot']

# e2_ligase = pd.read_csv(local_data_path + 'raw/E2_HPA.tsv', sep = '\t')
# e2_ligase = e2_ligase[e2_ligase['RNA cell line specificity'] == 'Low cell line specificity']
# e2_ligase = e2_ligase.loc[e2_ligase[e2_ligase['Subcellular main location'].notna()].index,:]
# e2_ligase = e2_ligase.loc[[i for i in e2_ligase.index if ('Cytosol' in e2_ligase.loc[i, 'Subcellular main location'])], :]
# # e2_ligase = e2_ligase[e2_ligase['Subcellular main location'] == 'Cytosol']
# top_gene_idx = e2_ligase.iloc[:, e2_ligase.columns.tolist().index('Tissue RNA - adipose tissue [NX]'):].mean(axis = 1).sort_values(ascending = False).index.tolist()[0]
# e2_uniprot_id = e2_ligase.loc[top_gene_idx, 'Uniprot']
# # DON'T DELETE------------------------------------------------

USP5, UBA1, UBE2D3, STUB1 = ['HGNC:12628'], ['HGNC:12469'], ['HGNC:12476'], ['HGNC:11427']
UB_ligases = UBA1 + UBE2D3 + STUB1
charged_trna_map = {v.id.split('_')[2]: v for v in charged_trna_metabolites}
psim_me = pd.read_csv(local_data_path + 'processed/psim_me.csv', index_col = 0)

proteasome_structural = ['HGNC:9554', 'HGNC:9560', 'HGNC:9557', 'HGNC:9556', 'HGNC:9564', 'HGNC:9565', 'HGNC:9558',
                        'HGNC:9566', 'HGNC:9567']
proteasome_ubiquitin = ['HGNC:9559', 'HGNC:15759', 'HGNC:16889', 'HGNC:9561', 'HGNC:12612', 'HGNC:19678']
proteasome_atpase = ['HGNC:9548', 'HGNC:9547', 'HGNC:9551', 'HGNC:9553', 'HGNC:9549', 'HGNC:9552']
proteasome_machinery = proteasome_structural + proteasome_ubiquitin + proteasome_atpase

seq_amino_acid_map_m = {aa_code: human_model.metabolites.get_by_id(met_obj.id.replace('[c]', '[m]')) for aa_code, met_obj in seq_amino_acid_map_c.items()}


# mitochondria

TOM = ['HGNC:31369', 'HGNC:34528', 'HGNC:21648', 'HGNC:20947', 'HGNC:18002', 'HGNC:18001', 'HGNC:11985']
# HGNC's tim23 already contains PAM
TIM23_PAM = pd.read_csv(local_data_path + 'raw/tim23_complex.csv',  index_col = None, skiprows = [0])['HGNC ID (gene)'].tolist()
HSP70_m = ['HGNC:5244'] # mitocondrial version
OXA = ['HGNC:8526'] # inner membrane transport

mLON, iAAA   = ['HGNC:9479'], ['HGNC:12843']# mitochondrial proteases
#mAAA =  ['HGNC:315', 'HGNC:11237'] 
HSP70_c, HSP40_c  = ['HGNC:5233'], ['HGNC:5229']

# peroxisome
seq_amino_acid_map_x = {aa_code: human_model.metabolites.get_by_id(met_obj.id.replace('[c]', '[x]')) for aa_code, met_obj in seq_amino_acid_map_c.items()}
PEX5, L_PEX5 = ['HGNC:9719'], 639 # Uniprot and PSIM_ME agree on this number
peroxins = ['HGNC:22965', 'HGNC:8859', 'HGNC:8850', 'HGNC:8856', 'HGNC:8855'] + PEX5
AWP1 = ['HGNC:30164']
LONP2 = ['HGNC:20598']

In [3]:
def make_protein_metabolite(id_, amino_acid_counts, L_protein, compartment):
    if compartment != 'c':
        raise ValueError('Must add this compartment to make_protein_metabolite function')
    protein_metabolite = cobra.Metabolite(id_ + '_protein[' + compartment + ']')
    protein_metabolite.compartment = compartment
    
    
    elements = {'C': 0, 'H': 0, 'N': 0, 'O': 0, 'S': 0}
    for aa_code, aa_count in amino_acid_counts.items():
        aa_elements = seq_amino_acid_map_c[aa_code].elements
        for element in aa_elements:
            elements[element] += aa_count*aa_elements[element]
    
    # peptide bond formation
    elements['H'] -= 2*(L_protein-1)
    elements['O'] -= 1*(L_protein-1)
    
    protein_metabolite.elements = elements
    protein_metabolite.charge = sum([seq_amino_acid_map_c[aa_code].charge*aa_count for aa_code, aa_count in amino_acid_counts.items()])
    return protein_metabolite

def translate_protein_cytosolic(gene_info):    

    # peptide bond formation: https://d1j63owfs0b5j3.cloudfront.net/pop-quiz/answerImage/Amino-Acid-1-popquiz.png
    # tRNA amino acide release: https://rnajournal.cshlp.org/content/14/8/1526/F1.expansion.html

    rxn = {charged_trna_map[aa_code]: -aa_count for aa_code, aa_count in gene_info.amino_acid_counts.items()} # tRNA consumption
    rxn[modified_trna_transcript_c] = gene_info.L_protein
    rxn[h2o_c] = -gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h_c] = gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h2o_c] += gene_info.L_protein - 1 # peptide bond formation (hydrolysis)
    
    # gtp hydrolysis per aa added
    rxn[ntp_map_c['G']] = -gene_info.L_protein 
    rxn[h2o_c] -= gene_info.L_protein
    rxn[ndp_map_c['G']] = gene_info.L_protein
    rxn[pi_c] = gene_info.L_protein
    rxn[h_c] += gene_info.L_protein
    

    rxn_c = rxn.copy()
    unfolded_protein_c = make_protein_metabolite(id_ = gene_info.hgnc_id + '_unfolded', 
                amino_acid_counts = gene_info.amino_acid_counts, L_protein = gene_info.L_protein,
                compartment = 'c')
    rxn_c[unfolded_protein_c] = 1
    
    translation_elongation = cobra.Reaction(gene_info.hgnc_id + '_CYTOSOLIC_TRANSLATION_ELONGATION')
    translation_elongation.subsytem = 'Protein_Expression'
    translation_elongation.add_metabolites(rxn_c)

    translation_elongation.gene_reaction_rule = ' and '.join(translation_efs + ['ribosome']) # GPRs

    return translation_elongation, unfolded_protein_c

def fold_protein_cytosolic(gene_info, unfolded_protein_c):
    # extending proteostasis network in the future would be good
    # will need to make sure inputs to each compartment-specific reactions are at the correct folding stage
    # e.g., mitochondria currently takes unfolded protein, and in future we may want it to take a partially folded
    
    folded_protein_c = unfolded_protein_c.copy()
    folded_protein_c.id = folded_protein_c.id.replace('unfolded', 'folded')
    rxn = {unfolded_protein_c: -1, folded_protein_c: 1}
    protein_folding = cobra.Reaction(gene_info.hgnc_id + '_CYTOSOLIC_PROTEIN_FOLDING')
    protein_folding.subsytem = 'Protein_Expression'
    
    if gene_info.L_protein > 100: #chaperone assisted for larger proteins - https://www.nature.com/articles/nature10317
        rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*proteolysis_translocation_atp_cost, compartment = 'c')
        protein_folding.gene_reaction_rule = ' and '.join(HSP40_c + HSP70_c) # GPRs
    
    
    protein_folding.add_metabolites(rxn)

    
    
    return protein_folding, folded_protein_c

Ubiquitin expression

In [4]:
# UBC
ubc_psim = psim_me[psim_me['HGNC_ID'] == 'HGNC:12468'] # UBC
ubc_psim['Location'] = 'c'
ubc_info = gene_information(metabolic_model = human_model, hgnc_id = ubc_psim['HGNC_ID'].values.tolist()[0], 
                         premrna_seq=ubc_psim['PREMRNA_SEQ'].values.tolist()[0], 
                            mrna_seq=ubc_psim['MRNA_SEQ'].values.tolist()[0], 
                            protein_seq=ubc_psim['PROTEIN_SEQ'].values.tolist()[0],
                            polyA_length = round(ubc_psim['POLYA_LENGTH'].values.tolist()[0]))
ubc_info.get_final_locations(human_model, final_locations=['c'])
ubc_mrna_expression_reactions = bm.mrna_expression(ubc_info)

# ubiquitin monomer
single_ubiquitin_sequence = ubc_info.protein_seq[:76]
monoub_aa_counts = {k: single_ubiquitin_sequence.count(k) for k in amino_acids}
L_monoub = len(single_ubiquitin_sequence)
n_ub_monomers = ubc_info.protein_seq.count(single_ubiquitin_sequence)
ub_c = make_protein_metabolite(id_ = 'ubiquitin_monomer', amino_acid_counts = monoub_aa_counts,
                               L_protein = L_monoub, compartment = 'c')

# monomerization from ubc polyub
# amino_acid_counts_ubc = {k: ubc_info.protein_seq.count(k) for k in amino_acids}
# L_ubc = len(ubc_info.protein_seq)

ubc_translation_reaction_cytosolic, ubc_c = translate_protein_cytosolic(ubc_info)

ubiquitin_monomerization_ubc = cobra.Reaction(ubc_info.hgnc_id + '_monomerization')
ubiquitin_monomerization_ubc.subsytem = 'Protein_Expression'
rxn = {ubc_c:-1, ub_c: n_ub_monomers, seq_amino_acid_map_c[ubc_info.protein_seq[n_ub_monomers*76:]]: 1, 
      h2o_c: -n_ub_monomers}
ubiquitin_monomerization_ubc.add_metabolites(rxn)
ubiquitin_monomerization_ubc.gene_reaction_rule = USP5[0]

# UBB
ubb_psim = psim_me[psim_me['HGNC_ID'] == 'HGNC:12463'] # UBB
ubb_psim['Location'] = 'c'
ubb_info = gene_information(metabolic_model = human_model, hgnc_id = ubb_psim['HGNC_ID'].values.tolist()[0], 
                         premrna_seq=ubb_psim['PREMRNA_SEQ'].values.tolist()[0], 
                            mrna_seq=ubb_psim['MRNA_SEQ'].values.tolist()[0], 
                            protein_seq=ubb_psim['PROTEIN_SEQ'].values.tolist()[0],
                            polyA_length = round(ubb_psim['POLYA_LENGTH'].values.tolist()[0]))
ubb_info.get_final_locations(human_model, final_locations=['c'])
ubb_mrna_expression_reactions = bm.mrna_expression(ubb_info)

# amino_acid_counts_ubb = {k: ubb_info.protein_seq.count(k) for k in amino_acids}
# L_ubb = len(ubb_info.protein_seq)

ubb_translation_reaction_cytosolic, ubb_c = translate_protein_cytosolic(ubb_info)

# monomerization from ubb polyub
n_ub_monomers = ubb_info.protein_seq.count(single_ubiquitin_sequence)
ubiquitin_monomerization_ubb = cobra.Reaction(ubb_info.hgnc_id + '_monomerization')
ubiquitin_monomerization_ubb.subsytem = 'Protein_Expression'
rxn = {ubb_c:-1, ub_c: n_ub_monomers, seq_amino_acid_map_c[ubb_info.protein_seq[n_ub_monomers*76:]]: 1, 
      h2o_c: -n_ub_monomers}
ubiquitin_monomerization_ubb.add_metabolites(rxn)
ubiquitin_monomerization_ubc.gene_reaction_rule = USP5[0]

# breakdown of the polyubiquitin cleaved from proteins in ubiquitin-proteasome pathway
polyub_aa_counts = {aa_code: aa_count*n_ub for aa_code, aa_count in monoub_aa_counts.items()}
polyub_c = make_protein_metabolite(id_ = 'cleaved_polyubiquitin_moiety', amino_acid_counts = polyub_aa_counts,
                               L_protein = L_monoub*n_ub, compartment = 'c')
ubiquitin_monomerization_polyub = cobra.Reaction('polyubiquitin_monomerization')
ubiquitin_monomerization_polyub.subsytem = 'Protein_Expression'
rxn = {polyub_c:-1, ub_c: n_ub, h2o_c: -(n_ub-1)}
ubiquitin_monomerization_polyub.add_metabolites(rxn)
ubiquitin_monomerization_polyub.gene_reaction_rule = USP5[0]

# degradation
degradation_ub = cobra.Reaction('ubiquitin_monomer_degradation')
degradation_ub.subsytem = 'Protein_Expression'
rxn = {seq_amino_acid_map_c[aa_code]: aa_counts for aa_code, aa_counts in monoub_aa_counts.items()}
rxn[ub_c] = -1
rxn[h2o_c] =  -(L_monoub-1)
# atp hydrolysis for translocation/unfolding by 26S - known 1 ATP per 2 residues - https://www.nature.com/articles/s41586-018-0736-4
rxn = hydrolyze_atp(rxn, n_atp = L_monoub/2, compartment = 'c')

degradation_ub.add_metabolites(rxn)
degradation_ub.gene_reaction_rule = ' and '.join(proteasome_machinery)

ub_reactions = ubc_mrna_expression_reactions + [ubc_translation_reaction_cytosolic] 
ub_reactions += ubb_mrna_expression_reactions + [ubb_translation_reaction_cytosolic]
ub_reactions += [ubiquitin_monomerization_ubc, ubiquitin_monomerization_ubb, ubiquitin_monomerization_polyub, degradation_ub]

/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:3 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:35 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


# Cytoplasmic Degradation

In [5]:
def protein_polyubiquitination(gene_info, protein_metabolite, compartment):
    if compartment == 'c':
        if protein_metabolite.compartment != 'c':
            raise ValueError('Compartment mismatch for polyubiquitination')
    
        polyubiquitinate_protein = cobra.Reaction(protein_metabolite.id + '_CYTOPLASMIC_POLYUBIQUITINATION')
        polyubiquitinate_protein.subsytem = 'Protein_Expression'


        polyu_protein_aa_counts = gene_info.amino_acid_counts.copy()
        for aa_code,aa_counts in monoub_aa_counts.items():
            if aa_code in polyu_protein_aa_counts:
                polyu_protein_aa_counts[aa_code] += aa_counts*n_ub
            else: 
                polyu_protein_aa_counts[aa_code] = aa_counts*n_ub

        polyub_protein_c = make_protein_metabolite(id_ = protein_metabolite.id + '_polyub', 
                           amino_acid_counts = polyu_protein_aa_counts, L_protein = gene_info.L_protein + (L_monoub*n_ub),
                           compartment = 'c') 
        rxn = {protein_metabolite: -1, ub_c: -n_ub, polyub_protein_c:1, h2o_c: n_ub}
        # 1 ATP hydrolysis per ubiquitin monomer added (https://link.springer.com/article/10.1007/s10637-020-00894-6)
        rxn = hydrolyze_atp(rxn, n_atp = n_ub, compartment = 'c')


        polyubiquitinate_protein.add_metabolites(rxn)
        polyubiquitinate_protein.gene_reaction_rule = ' and '.join(UB_ligases)
        return polyubiquitinate_protein, polyub_protein_c
    else:
        raise ValueError('Current compartment does not have polyubiquitination')

def proteasomal_degradation(gene_info, protein_metabolite, polyub_protein_metabolite, compartment):
    if compartment == 'c':
        if (protein_metabolite.compartment != 'c') or (polyub_protein_metabolite.compartment != 'c'):
            raise ValueError('Compartment mismatch for cytoplasmic proteasomal degradation')
        
        
        deubiquitination = cobra.Reaction(protein_metabolite.id + '_CYTOPLASMIC_DEUBIQUITINATION')
        deubiquitination.subsytem = 'Protein_Expression'
        deubiquitination.add_metabolites({polyub_protein_metabolite: -1, h2o_c: -1, protein_metabolite: 1, polyub_c: 1})
        deubiquitination.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation = cobra.Reaction(protein_metabolite.id + '_CYTOPLASMIC_PROTEASOMAL_DEGRADATION')
        protein_degradation.subsytem = 'Protein_Expression'
        rxn = {seq_amino_acid_map_c[aa_code]: aa_counts for aa_code, aa_counts in gene_info.amino_acid_counts.items()}
        rxn[polyub_protein_metabolite], rxn[h2o_c], rxn[polyub_c] = -1, -gene_info.L_protein, 1
        # atp hydrolysis for translocation/unfolding  - known 1 ATP per 2 residues - https://www.nature.com/articles/s41586-018-0736-4
        L_polub_protein = (gene_info.L_protein + (L_monoub*n_ub)) 
        rxn = hydrolyze_atp(rxn, n_atp = L_polub_protein/2, compartment = 'c')


        protein_degradation.add_metabolites(rxn)
        protein_degradation.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation_reactions = [deubiquitination, protein_degradation]

        return protein_degradation_reactions
    else:
        raise ValueError('Current compartment does not have proteasomal degradation')

# Mitochondrial Trasport/Degradation


In [6]:
# i is intermembrane space, but called inner in compartments BIGG
# stick to notation and use inner instead of inter in reaction naming

def transport_mitochondrial_matrix(gene_info, unfolded_protein_c):
    # transport and folding
    if unfolded_protein_c.compartment != 'c':
        raise ValueError('Only cytoplasmic proteins can be transported to mitochondrial matrix')
    
    mitochondrial_matrix_transport = cobra.Reaction(gene_info.hgnc_id + '_MITOCHONDRIAL_MATRIXtn')
    mitochondrial_matrix_transport.subsytem = 'Protein_Expression'
    pre_protein_m = unfolded_protein_c.copy()
    pre_protein_m.id = pre_protein_m.id.replace('[c]', '[m]')
    pre_protein_m.compartment = 'm'
    pre_protein_m.id = pre_protein_m.id.replace('unfolded', 'folded_pre')
    
    rxn = {unfolded_protein_c: -1, pre_protein_m: 1}
    # ATP hydrolysis for transport, assums 1 ATP consumed per 2 residues
    rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*transport_translocation_atp_cost, compartment = 'm')
    
    
    mitochondrial_matrix_transport.add_metabolites(rxn)
    mitochondrial_matrix_transport.gene_protein_rule = ' and '.join(TOM + TIM23_PAM + HSP70_m)
    
    return mitochondrial_matrix_transport, pre_protein_m

def mitochondrial_matrix_protein_processing(gene_info, pre_protein_m):
    # implement this in the future: cleavage of MTS (and degradation of MTS)
    processed_protein_m, aa_counts_processed_m, L_processed_protein_m = pre_protein_m, gene_info.amino_acid_counts.copy(), gene_info.L_protein
    process_mitochondrial_matrix_protein = None
    return process_mitochondrial_matrix_protein, processed_protein_m, aa_counts_processed_m, L_processed_protein_m


def degrade_mitochondrial_protein(gene_info, protein_metabolite, compartment, L_protein, amino_acid_counts):
    rxn = {seq_amino_acid_map_m[aa_code]: aa_counts for aa_code, aa_counts in amino_acid_counts.items()}
    rxn[protein_metabolite], rxn[h2o_m] = -1, -(L_protein-1)
    
    if compartment == 'm':
        mitochondrial_degradation = cobra.Reaction(gene_info.hgnc_id + '_MITOCHONDRIAL_MATRIX_DEGRADATION')
        mitochondrial_degradation.gene_protein_rule = mLON[0]
        
        # ATP hydrolysis by LON: 2 ATP per residue - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2518814/
        rxn = hydrolyze_atp(rxn, n_atp = L_protein*2, compartment = 'm')
        

    elif compartment == 'i':
        mitochondrial_degradation = cobra.Reaction(gene_info.hgnc_id + '_INNER_MITOCHONDRIAL_DEGRADATION')
        mitochondrial_degradation.gene_protein_rule = iAAA[0]#' and '.join(mAAA + iAAA)
        
        # ATP hydrolysis by m/i-AAA: 1 ATP per 2 residues -- no source, assumes same as 26S proteasome
        rxn = hydrolyze_atp(rxn, n_atp = L_protein*proteolysis_translocation_atp_cost, compartment = 'i')
  
    mitochondrial_degradation.subsytem = 'Protein_Expression'
    mitochondrial_degradation.add_metabolites(rxn)
    
    return mitochondrial_degradation

def transport_mitochondrial_inter(gene_info, processed_protein_m):
    # upper left Fig 12-29 https://www.ncbi.nlm.nih.gov/books/NBK26828/ 
    # import to matrix then re-export to inter membrane space
    
    if processed_protein_m.compartment != 'm':
        raise ValueError('Only the mechanism of mitochondrial matrix import and re-export to inter membrane is considered')
    
    mitochondrial_inter_transport = cobra.Reaction(gene_info.hgnc_id + '_MITOCHONDRIAL_INNERtn')
    mitochondrial_inter_transport.subsytem = 'Protein_Expression'
    pre_protein_i = processed_protein_m.copy()
    pre_protein_i.id = processed_protein_m.id.replace('[m]', '[i]')
    pre_protein_i.compartment = 'i'
    
    rxn = {processed_protein_m: -1, pre_protein_i: 1}
    
    mitochondrial_inter_transport.add_metabolites(rxn)
    mitochondrial_inter_transport.gene_protein_rule = OXA[0]
    
    return mitochondrial_inter_transport, pre_protein_i

def mitochondrial_inter_protein_processing(gene_info, pre_protein_i):
    # implement this in the future: cleavage of secondary sequence (and degradation)
    processed_protein_i, aa_counts_processed_i, L_processed_protein_i = pre_protein_i, gene_info.amino_acid_counts.copy(), gene_info.L_protein
    process_mitochondrial_matrix_protein = None
    return process_mitochondrial_matrix_protein, processed_protein_i, aa_counts_processed_i, L_processed_protein_i

def get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments):
    mitochondrial_matrix_transport, pre_protein_m = transport_mitochondrial_matrix(gene_info, unfolded_protein_c)
    process_mitochondrial_matrix_protein, processed_protein_m, aa_counts_processed_m, L_processed_protein_m = mitochondrial_matrix_protein_processing(gene_info, pre_protein_m)
    
    mitochondrial_reactions = [mitochondrial_matrix_transport]
    if process_mitochondrial_matrix_protein != None:
        mitochondrial_reactions += [process_mitochondrial_matrix_protein]
    
    if 'm' in compartments:
        mitochondrial_matrix_degradation = degrade_mitochondrial_protein(gene_info, protein_metabolite = processed_protein_m, compartment = 'm', L_protein = L_processed_protein_m, amino_acid_counts = aa_counts_processed_m)
        mitochondrial_reactions += [mitochondrial_matrix_degradation]
    if 'i' in compartments:
        mitochondrial_inter_transport, pre_protein_i = transport_mitochondrial_inter(gene_info, processed_protein_m)
        process_mitochondrial_inter_protein, processed_protein_i, aa_counts_processed_i, L_processed_protein_i = mitochondrial_inter_protein_processing(gene_info, pre_protein_i)
        if process_mitochondrial_matrix_protein != None:
                mitochondrial_reactions += [process_mitochondrial_inter_protein]        
        mitochondrial_inter_degradation = degrade_mitochondrial_protein(gene_info, protein_metabolite = processed_protein_i, compartment = 'i', L_protein = L_processed_protein_i, amino_acid_counts = aa_counts_processed_i)
        mitochondrial_reactions += [mitochondrial_inter_transport, mitochondrial_inter_degradation]

    return mitochondrial_reactions

# Peroxisomal

In [7]:
def transport_peroxisome(gene_info, folded_protein_c):
    if folded_protein_c.compartment != 'c':
        raise ValueError('Only cytoplasmic proteins can be transported to mitochondrial matrix')
    
    peroxisomal_transport = cobra.Reaction(gene_info.hgnc_id + '_PEROXISOMEtn')
    peroxisomal_transport.subsytem = 'Protein_Expression'
    protein_x = folded_protein_c.copy()
    protein_x.id = protein_x.id.replace('[c]', '[x]')
    protein_x.compartment = 'x'
    
    rxn = {folded_protein_c: -1, protein_x: 1}
    # ATP hydrolysis for transport--translocation of protein, export of PEX5S receptor
    rxn = hydrolyze_atp(rxn, n_atp = (gene_info.L_protein+L_PEX5)*transport_translocation_atp_cost, 
                        compartment = 'x')
    
    
    peroxisomal_transport.add_metabolites(rxn)
    peroxisomal_transport.gene_protein_rule = ' and '.join(peroxins + AWP1)
    
    return peroxisomal_transport, protein_x

def degrade_peroxisomal_protein(gene_info, protein_x):
    
    
    peroxisomal_degradation = cobra.Reaction(gene_info.hgnc_id + '_PEROXISOMAL_DEGRADATION')
    peroxisomal_degradation.subsytem = 'Protein_Expression'
    peroxisomal_degradation.gene_protein_rule = LONP2[0]

    rxn = {seq_amino_acid_map_x[aa_code]: aa_counts for aa_code, aa_counts in gene_info.amino_acid_counts.items()}
    rxn[protein_x], rxn[h2o_x] = -1, -(gene_info.L_protein-1)
    # ATP hydrolysis by LON: 2 ATP per residue - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2518814/
    rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*2, compartment = 'x')
    peroxisomal_degradation.add_metabolites(rxn)
    
    return peroxisomal_degradation

def get_peroxisomal_reactions(gene_info, folded_protein_c):
    peroxisomal_transport, protein_x = transport_peroxisome(gene_info, folded_protein_c)
    peroxisomal_degradation = degrade_peroxisomal_protein(gene_info, protein_x)
    
    return [peroxisomal_transport, peroxisomal_degradation]

In [8]:
def get_protein_expression_reactions(gene_info):
    # after transport, expand these to secretory pathways
    protein_expression_reactions = list()
    
    # cytoplasmic translation
    if 'Cytosolic Tranport' in gene_info.final_locations.values() or gene_info.L_protein <= 160: 
        translation_elongation_c, unfolded_protein_c = translate_protein_cytosolic(gene_info)
        protein_expression_reactions.append(translation_elongation_c)
    
    #UPDATE if statement: cytoplasmic folding and degradation    
    if 'c' in gene_info.final_locations.keys() or 'x' in gene_info.final_locations.keys():
        protein_folding_cytosolic, folded_protein_c = fold_protein_cytosolic(gene_info, unfolded_protein_c)
        
        polyubiquitinate_folded_protein_c, polyub_protein_c = protein_polyubiquitination(gene_info, protein_metabolite = folded_protein_c, compartment = 'c')
        protein_degradation_reactions_folded_c = proteasomal_degradation(gene_info, protein_metabolite = folded_protein_c, 
                                          polyub_protein_metabolite = polyub_protein_c, compartment = 'c')
        protein_expression_reactions += [protein_folding_cytosolic, polyubiquitinate_folded_protein_c] + protein_degradation_reactions_folded_c
    
        if 'x' in gene_info.final_locations.keys():
            protein_expression_reactions += get_peroxisomal_reactions(gene_info, folded_protein_c)
            
            
    if 'i' in gene_info.final_locations.keys(): # no folding for i
        polyubiquitinate_unfolded_protein_c, polyub_protein_c = protein_polyubiquitination(gene_info, protein_metabolite = unfolded_protein_c, compartment = 'c')
        protein_degradation_reactions_unfolded_c = proteasomal_degradation(gene_info, protein_metabolite = unfolded_protein_c, 
                                          polyub_protein_metabolite = polyub_protein_c, compartment = 'c')
        protein_expression_reactions += [polyubiquitinate_unfolded_protein_c] + protein_degradation_reactions_unfolded_c

    
    # mitochondrial transport and degradation ('i' and 'm')
    if ('m' in gene_info.final_locations.keys()) or ('i' in gene_info.final_locations.keys()):
        if ('m' in gene_info.final_locations.keys()) and ('i' in gene_info.final_locations.keys()):
            mitochondrial_reactions = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['m','i'])
        elif 'm' in gene_info.final_locations.keys():
            mitochondrial_reactions = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['m'])
        elif 'i' in gene_info.final_locations.keys():
            mitochondrial_reactions = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['i'])
        protein_expression_reactions += mitochondrial_reactions
    
        
    return protein_expression_reactions

In [9]:
# gene_info.final_locations = {'c': 'Cytosolic Tranport', 'm': 'Cytosolic Tranport', 
#                              'i': 'Cytosolic Tranport'}
# mit_reactions = get_protein_expression_reactions(gene_info)
# mit_mod = cobra.Model('mitochondrial_expression')
# mit_mod.add_reactions(mit_reactions)
# import escher
# builder = escher.Builder(model = mit_mod)

In [10]:
sp_dict = {1: True, 0: False, float('nan'): False}
ptm_cols = ['DSB', 'GPI', 'NG', 'OG']
ptm_keys = list(allowed_ptms.keys())

gene1_id = human_model.genes[0].id


idx  = psim_me[psim_me['HGNC_ID'] == gene1_id].index
ptms_ = dict(zip(ptm_keys, psim_me.loc[idx, ptm_cols].iloc[0,:].tolist()))
ptms_ = {k:v for k,v in ptms_.items() if v != 0 and not pd.isna(v)}
fl = psim_me.loc[idx, 'Location'].tolist()[0]

pm,m,p = psim_me.loc[idx, 'PREMRNA_SEQ'].tolist()[0], psim_me.loc[idx, 'MRNA_SEQ'].tolist()[0], psim_me.loc[idx, 'PROTEIN_SEQ'].tolist()[0]

sp = psim_me.loc[idx, 'SP'].tolist()[0]
if pd.isna(sp):
    sp = 0
sp = sp_dict[sp]
tmd = psim_me.loc[idx,'TMD'].tolist()[0]
if pd.isna(tmd):
    tmd = 0
polyA_length_ = psim_me.loc[idx, 'POLYA_LENGTH'].tolist()[0]
gene_info = gene_information(metabolic_model = human_model, hgnc_id = gene1_id, 
                         premrna_seq=pm, mrna_seq=m, protein_seq=p,
                         ptms = ptms_, tmd = tmd, sp = sp, 
                        keff = None, polyA_length = polyA_length_, n_introns= None)
gene_info.get_final_locations(human_model)
mrna_expression_reactions = bm.mrna_expression(gene_info)

../scripts/gene_information.py:135 UserWarning: No keff specified for this enzyme, will assume a value in model building
../scripts/gene_information.py:217 UserWarning: Final location is part of secretory pathway, but no signal peptide indicated.Non canonical secretion is not considered currently. Changing sp to True
../scripts/gene_information.py:205 UserWarning: Signal peptides not considered for these compartments


In [11]:
gene_info.final_locations = {'x': 'Cytosolic Tranport'}
mit_reactions = get_protein_expression_reactions(gene_info)

In [12]:
[r.check_mass_balance() for r in mit_reactions]

[{}, {}, {}, {}, {}, {}, {}]